# 07 - MARP yeniden çekim politikası
Her aday düzeltme eylemi için gözlenen mekanik hata azalmasını öğrenir ve beklenen hata azalması en yüksek eylemi önerir.

In [ ]:
import sys, zipfile
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    artifacts_root = Path('/content/drive/MyDrive/PhysioLoad_AI_Project')
    code_root = Path('/content/physioload-ai')
    if not (code_root / 'physioload_env.py').exists():
        code_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(artifacts_root / 'code' / 'physioload-source.zip') as package: package.extractall(code_root)
else:
    artifacts_root = None
    here = Path.cwd().resolve()
    code_root = next(p for p in (here, *here.parents) if (p / 'physioload_env.py').exists())
sys.path.insert(0, str(code_root))
from physioload_env import setup_environment
env = setup_environment(root=code_root, artifacts_root=artifacts_root)
print(env.summary())

In [ ]:
import joblib, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

rows_path = env.predictions_dir / 'marp_action_outcomes.csv'
if not rows_path.exists():
    raise FileNotFoundError(f'Kontrollü yeniden çekim sonuçları gerekli: {rows_path}')
rows = pd.read_csv(rows_path)
numeric = ['visibility','pose_confidence','view_deviation_deg','geometry_penalty','error_before']
categorical = ['action']
target = 'error_reduction'
train = rows[rows.split == 'train']
validation = rows[rows.split == 'validation']
preprocess = ColumnTransformer([('num', StandardScaler(), numeric), ('action', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical)])
policy = Pipeline([('preprocess', preprocess), ('regressor', HistGradientBoostingRegressor(random_state=42))])
policy.fit(train[numeric + categorical], train[target])
print('train rows:', len(train), 'validation rows:', len(validation))

In [ ]:
from sklearn.metrics import mean_absolute_error
prediction = policy.predict(validation[numeric + categorical])
print('validation MAE:', mean_absolute_error(validation[target], prediction))
policy_path = env.calibrators_dir / f'marp_{env.processed_version}.joblib'
joblib.dump({'model': policy, 'numeric': numeric, 'categorical': categorical, 'processed_version': env.processed_version}, policy_path)
print('saved:', policy_path)

MARP'nin bilimsel katkısı ancak standart kamera talimatı ve genel yeniden çekim baseline'larından daha büyük hata azalması kontrollü testte gösterilirse savunulabilir.